In [6]:
import sys
sys.path.insert(0, '..')

import sqlite3
import pandas as pd
from datetime import datetime
from src.data import (
    load_and_prepare, get_username_mappings,
    load_standings, load_model_predictions,
)
import pprint


In [3]:
df = load_standings()
USERNAME_TO_PLAYER, PLAYER_TO_USERNAME = get_username_mappings()

df['name'] = df['username'].apply(lambda text: USERNAME_TO_PLAYER.get(text,"unknown"))

df.head()

,date,tournament_slug,session,rank,username,title,country,rating,score,tie_break,wins,draws,byes,name
0,2014-10-28,-monthly-32-blitz-masters-443681,NaN,1.0,exoticprincess,GM,Georgia,3002,8.0,46.0,8.0,0.0,0.0,Baadur Jobava
1,2014-10-28,-monthly-32-blitz-masters-443681,NaN,2.0,Danesz22,IM,Romania,2289,7.0,33.0,7.0,0.0,0.0,Daniel Baratosi
2,2014-10-28,-monthly-32-blitz-masters-443681,NaN,3.0,GMG,GM,United States,2732,6.5,33.0,6.0,1.0,0.0,Andrey Gorovets
3,2014-10-28,-monthly-32-blitz-masters-443681,NaN,4.0,GeorgMeier,GM,Uruguay,3016,6.5,33.0,5.0,3.0,0.0,Georg Meier
4,2014-10-28,-monthly-32-blitz-masters-443681,NaN,5.0,GMNeiksans,GM,Latvia,2853,6.5,32.0,6.0,1.0,0.0,Arturs Neiksans


In [4]:
cct = df[df['date'].between('2025-09-01','2025-11-26')]

In [7]:
import numpy as np

DECAY = 0.85

# Sorted dates: index 0 = 2025-09-02 (highest weight), last = 2025-11-25 (lowest)
dates = sorted(cct['date'].dt.normalize().unique())
weights = np.array([DECAY ** i for i in range(len(dates))])

# Player x date attendance matrix (1 = attended, 0 = absent)
attended = pd.crosstab(cct['name'], cct['date'].dt.normalize()).reindex(columns=dates, fill_value=0)

# EWMA: weighted sum / total weight
ewma = (attended * weights).sum(axis=1) / weights.sum()

cct_ewma = ewma.to_dict()


# Pretty print without length truncation
pprint.pprint(cct_ewma, compact=False, width=120)

{'Aadit Bhatia': 0.2403533007081324,
 'Aadya Gowda': 0.1706301187211643,
 'Aadya Gupta': 0.22913254518649942,
 'Aamuktha Guntaka': 0.12328026077604119,
 'Aansh Nandan Nerurkar': 0.10478822165963501,
 'Aaradhya Das': 0.26552848195358414,
 'Aarav Dengla': 0.5146554705592814,
 'Aaron Bendayan Claros': 0.04649509063780762,
 'Aaron Mendes': 0.08906998841068975,
 'Aaron Zambrano': 0.02427072790725206,
 'Abdulkadir Mohamed Warsame': 0.08906998841068975,
 'Abhimanyu Puranik': 0.0757094901490863,
 'Abtin Atakhan': 0.6843342803658461,
 'Adam Gibała': 0.3224641770351491,
 'Adam Kozak': 0.033592702985816,
 'Adam Omarsson': 0.04649509063780762,
 'Adar TARHAN': 0.22849354961423238,
 'Adharsh K': 0.1706301187211643,
 'Aditya Kunal Patil': 0.052824525445195655,
 'Aditya Mittal': 0.0757094901490863,
 'Aditya Varun Gampa': 0.14503560091298964,
 'Adnan Sitnic': 0.37151538611981105,
 'Adrian Antonio Garcia Garcia': 0.2939103794972055,
 'Adrian Keatinge-Clay': 0.02427072790725206,
 'Adrian Miranda': 0.0890